# ETL del archivo en crudo `crew.parquet`

## Librerías

In [1]:
import os
import ast
import gc

import pandas as pd

## Extracción

In [2]:
url = "https://github.com/FranciscoHugoLezik/Movies_data/blob/main/credits/crew.parquet?raw=true"

crew = pd.read_parquet(
    url, 
    engine='fastparquet'
    )

In [3]:
crew.head()

,crew,id
0,"[{'credit_id': '52fe4284c3a36847f8024f49', 'de...",862
1,"[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de...",8844
2,"[{'credit_id': '52fe466a9251416c75077a89', 'de...",15602
3,"[{'credit_id': '52fe44779251416c91011acb', 'de...",31357
4,"[{'credit_id': '52fe44959251416c75039ed7', 'de...",11862


Valor en la columna 'crew' de la primera fila. Es una cadena con la forma de una lista de diccionarios.

In [4]:
crew['crew'].iloc[0]

'[{\'credit_id\': \'52fe4284c3a36847f8024f49\', \'department\': \'Directing\', \'gender\': 2, \'id\': 7879, \'job\': \'Director\', \'name\': \'John Lasseter\', \'profile_path\': \'/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f4f\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12891, \'job\': \'Screenplay\', \'name\': \'Joss Whedon\', \'profile_path\': \'/dTiVsuaTVTeGmvkhcyJvKp2A5kr.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f55\', \'department\': \'Writing\', \'gender\': 2, \'id\': 7, \'job\': \'Screenplay\', \'name\': \'Andrew Stanton\', \'profile_path\': \'/pvQWsu0qc8JFQhMVJkTHuexUAa1.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f5b\', \'department\': \'Writing\', \'gender\': 2, \'id\': 12892, \'job\': \'Screenplay\', \'name\': \'Joel Cohen\', \'profile_path\': \'/dAubAiZcvKFbboWlj7oXOkZnTSu.jpg\'}, {\'credit_id\': \'52fe4284c3a36847f8024f61\', \'department\': \'Writing\', \'gender\': 0, \'id\': 12893, \'job\': \'Screenplay\', \'name\': \'A

La columnas estan completas.

In [5]:
crew.isnull().sum()

crew    0
id      0
dtype: int64

## Transformación

### Borrar duplicados en la columna 'id'

Hay valores duplicados.

In [6]:
crew['id'].duplicated(keep='first').sum()

44

Se eliminan los duplicados.

In [7]:
crew.drop_duplicates(
    subset='id', 
    inplace=True
    )

Hay valores únicos.

In [8]:
crew['id'].duplicated(keep='first').sum()

0

### Renombrar el nombre de la columna 'id' por 'movie_id'

Se hace esto porque dentro de los valores anidados hay una clave llamada 'id' y para, mas adelante, poder hacer un join con el dataset 'movies.parquet'.

In [9]:
crew.rename(
    columns={'id': 'movie_id'}, 
    inplace=True
    )

Se cambio el nombre.

In [10]:
crew.columns

Index(['crew', 'movie_id'], dtype='object')

### Eliminar listas vacias en la columna 'crew'

Se van a eliminar listas vacías de la columna 'crew' para achicar el tamaño del dataset.

Hay listas vacías.

In [11]:
crew[crew['crew'] == "[]"]

,crew,movie_id
189,[],56088
614,[],123505
635,[],339428
661,[],318177
711,[],365371
...,...,...
45236,[],332543
45261,[],28469
45277,[],458618
45348,[],335251


In [12]:
len(crew[crew['crew'] == "[]"])

771

Se eliminan las listas vacías.

In [13]:
crew = crew.query('crew != "[]"')

Las listas vacías estan eliminadas.

In [14]:
len(crew[crew['crew'] == "[]"]) == 0

True

### Desanidar la columna 'crew'

Se convierte a las cadenas en listas de diccionarios.

In [15]:
crew['crew'] = crew['crew'].apply(
    ast.literal_eval)

Se separan los elementos de las listas en filas.

In [16]:
crew_en_filas = crew.explode('crew')

Se convierten las llaves en columnas.

In [17]:
crew_en_columnas = pd.json_normalize(
    crew_en_filas['crew'])

### Crear dataframe 'crew' con nuevas columnas

Se crea un dataframe con la columna 'movie_id' y las nuevas columnas.

In [18]:
crew = crew_en_filas.drop(
    columns='crew').join(
        crew_en_columnas)

Se eliminan los siguientes objetos para liberar memoria.

In [19]:
del crew_en_filas
del crew_en_columnas
gc.collect()

308

## Exploración

Se explora el dataframe.

In [20]:
crew

,movie_id,credit_id,department,gender,id,job,name,profile_path
0,862,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
0,862,52fe4284c3a36847f8024f49,Directing,2,7879,Director,John Lasseter,/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
...,...,...,...,...,...,...,...,...
45473,67758,52fe4368c3a36847f80520fb,Art,2,1966,Production Design,Jacques Bufnoir,None
45473,67758,52fe4368c3a36847f80520fb,Art,2,1966,Production Design,Jacques Bufnoir,None
45474,227506,52fe4368c3a36847f8052101,Writing,2,27722,Screenplay,Érik Orsenna,None
45474,227506,52fe4368c3a36847f8052101,Writing,2,27722,Screenplay,Érik Orsenna,None


Primera fila.

In [21]:
crew.iloc[0]

movie_id                                     862
credit_id               52fe4284c3a36847f8024f49
department                             Directing
gender                                         2
id                                          7879
job                                     Director
name                               John Lasseter
profile_path    /7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg
Name: 0, dtype: object

Se obtiene una informacion general.

In [22]:
crew.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 463836 entries, 0 to 45475
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   movie_id      463836 non-null  int64 
 1   credit_id     463836 non-null  object
 2   department    463836 non-null  object
 3   gender        463836 non-null  int64 
 4   id            463836 non-null  int64 
 5   job           463836 non-null  object
 6   name          463836 non-null  object
 7   profile_path  106386 non-null  object
dtypes: int64(3), object(5)
memory usage: 31.8+ MB


La columna 'name' esta completa.

In [23]:
crew['name'].isnull().sum()

0

## Transformación de los datos desanidados

### Eliminar las columnas innecesarias

Se las elimina porque son inutiles para la funcion get_director o para cualquier busqueda en la que se busque personas de otro trabajo o departamento.

Columnas innecesarias.

In [24]:
innecesarias = [
    'credit_id', 
    'gender', 
    'id', 
    'profile_path'
]

Las columnas innecesarias son eliminadas.

In [25]:
crew.drop(
    columns=innecesarias, 
    inplace=True
    )

Las columnas innecesarias estan eliminadas.

In [26]:
set(crew.columns).isdisjoint(set(innecesarias))

True

Se eliminan las columnas inncesarias de la memoria.

In [ ]:
del crew['credit_id']
del crew['gender']
del crew['id']
del crew['profile_path']
gc.collect()

Se ven la columnas que quedan.

In [27]:
for columna in crew.columns:
    print(columna)

movie_id
department
job
name


### Cambiar el tipo de la columna 'movie_id'

La columna 'movie_id' tiene etiquetas. Entonces se lo cambia al tipo object. El resto de los datos tienen el tipo correcto que es object.

Tipos de las columnas:

In [28]:
crew.dtypes

movie_id       int64
department    object
job           object
name          object
dtype: object

* Columna 'movie_id'

In [29]:
crew['movie_id'].dtype

dtype('int64')

In [30]:
crew['movie_id'] = crew['movie_id'].astype(str)

In [31]:
crew['movie_id'].dtype

dtype('O')

Tipos de las columnas con la modificación:

In [32]:
crew.dtypes

movie_id      object
department    object
job           object
name          object
dtype: object

### Resetear el índice

El índice actual:

In [33]:
crew.index

Int64Index([    0,     0,     0,     0,     0,     0,     0,     0,     0,
                0,
            ...
            45472, 45472, 45473, 45473, 45473, 45473, 45473, 45474, 45474,
            45475],
           dtype='int64', length=463836)

Se resetea el índice:

In [34]:
crew.reset_index(
    drop=True, 
    inplace=True
    )

El índice actualizado:

In [35]:
crew.index

RangeIndex(start=0, stop=463836, step=1)

### Última revisión

In [36]:
crew.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 463836 entries, 0 to 463835
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   movie_id    463836 non-null  object
 1   department  463836 non-null  object
 2   job         463836 non-null  object
 3   name        463836 non-null  object
dtypes: object(4)
memory usage: 14.2+ MB


## Carga

In [37]:
ruta_actual = os.getcwd()

ruta_actual

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\notebooks\\ETL'

In [38]:
ruta_del_proyecto = os.path.dirname(
    os.path.dirname(
        ruta_actual))

ruta_del_proyecto

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas'

In [39]:
ruta_a_exportar = os.path.join(
    ruta_del_proyecto, 
    'data', 
    'ETL', 
    'crew.parquet')

ruta_a_exportar

'c:\\Users\\franc\\Desktop\\Proyecto_Peliculas\\data\\ETL\\crew.parquet'

In [40]:
crew.to_parquet(ruta_a_exportar)

Se elimina el dataframe para liberar memoria.

In [41]:
del crew
gc.collect()

15